In [1]:
# ==========================================================================
# D52 자율 프로젝트 : 📰 뉴스 섹션 자동 분류 API
#   기사 제목 -> KLUE-YNAT 7개 섹션(IT과학·경제·사회·생활문화·세계·스포츠·정치) 추천
#   확신이 낮은 제목은 '편집자 검토 필요' 로 표시, 여러 제목은 한 번에 분류해 섹션별 집계
#
#   모델 : chunwoolee0/klue_ynat_roberta_base_model (revision refs/pr/2, 443MB)
#   환경 : WSL2 Ubuntu, Python 3.12, CPU, torch 2.4.1, transformers 4.57.6
#   서비스 : FastAPI 8000번 (/docs = Swagger UI), Streamlit 8501번
#   확인 URL :
#     프론트 (Streamlit)      http://localhost:8501
#     백엔드 Swagger UI       http://localhost:8000/docs
#     백엔드 ReDoc 문서       http://localhost:8000/redoc
#     백엔드 헬스체크         http://localhost:8000/health
#     백엔드 섹션 목록        http://localhost:8000/labels
#     추론 (POST, 키 필요)    http://localhost:8000/predict , /predict/batch   (X-API-Key: test-key-001)
#
#   필수 구현 : ① FastAPI POST /predict + Pydantic 검증 + run_in_executor
#               ② API Key 인증(auth.py)  ③ Streamlit 화면  ④ 에러 처리(401/422/503/500)
#   평가 기준 : 서버 실행 / Swagger 추론 / 키 없으면 401 / 잘못된 입력 에러 메시지 / Streamlit 입출력
#   하지 않는 것 : 모델 학습, Docker 패키징, DB 연동
# ==========================================================================
# [0. 작업 폴더] app/, frontend/ 같은 상대 경로의 기준 폴더를 맞춥니다
#   - GitHub 레포(Serving/Serving01) : 노트북 옆에 app/ 이 있으므로 그대로 씁니다
#   - 로컬 작업본(notebooks/)        : 형제 폴더 ../my-project 로 이동합니다
import os
from pathlib import Path

here = Path.cwd()
if not (here / "app").exists() and (here.parent / "my-project").exists():
    os.chdir(here.parent / "my-project")

print("작업 폴더 :", os.getcwd())

작업 폴더 : /mnt/c/EPA1st/model-serving-course/my-project


In [2]:
# ==========================================================================
# [2. 모델 선택과 동작 확인] 서버 코드를 쓰기 전에 노트북에서 단건 추론부터 확인합니다
#   선택 기준 : 태스크가 명확한가 / 한국어 지원 / CPU 기준 500MB 이하 / pipeline() 으로 바로 사용 가능
#   선택 이유 :
#     - KLUE-RoBERTa-base 를 KLUE-YNAT(연합뉴스 제목 약 4.5만 건)로 미세조정한 모델
#     - id2label 에 한국어 섹션 이름이 들어 있어 후처리 매핑이 필요 없음
#   막혔던 점 : 원본 저장소는 pytorch_model.bin 만 있음
#              -> transformers 4.57 이 torch 2.6 미만에서 .bin 로드를 막음 (CVE-2025-32434)
#              -> 변환 봇이 올린 safetensors 판 PR 브랜치를 revision="refs/pr/2" 로 지정해 해결
# ==========================================================================
# 내 모델 : 한국어 뉴스 제목 -> KLUE-YNAT 7개 섹션 분류
# - 원본 저장소는 pytorch_model.bin 만 있어 torch 2.4.1 에서 읽히지 않습니다 (아래 설명 참고)
# - 허깅페이스 변환 봇이 올린 safetensors 판(refs/pr/2)을 revision 으로 지정합니다
import time

import torch
import transformers
from transformers import pipeline

print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)

t0 = time.perf_counter()
classifier = pipeline(
    "text-classification",
    model="chunwoolee0/klue_ynat_roberta_base_model",
    revision="refs/pr/2",
    top_k=None,      # 7개 섹션 확률 전부
    device=-1,       # CPU
)
print(f"로드 시간    : {time.perf_counter() - t0:.1f}초")
print("섹션 목록    :", list(classifier.model.config.id2label.values()))

result = classifier("코스피 2,800선 회복, 외국인 순매수 이어져")
print(result[0][:3])

/mnt/c/EPA1st/model-serving-course/.venv-wsl312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch        : 2.4.1+cpu
transformers : 4.57.6


Device set to use cpu


로드 시간    : 1.6초
섹션 목록    : ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']


[{'label': '경제', 'score': 0.9907235503196716}, {'label': '사회', 'score': 0.0021201884374022484}, {'label': '세계', 'score': 0.0020517981611192226}]


In [3]:
# [2. 모델 비교] 직접 만든 제목 14건(섹션별 2건)으로 정확도와 속도를 확인합니다
#   같은 태스크 모델 비교 결과 :
#     chunwoolee0/klue_ynat_roberta_base_model         14/14  라벨이 한국어 -> 선택
#     kakarooky/roberta-base-klue-ynat-classification  14/14  라벨이 LABEL_0~6 이라 매핑 필요
#     Doowon96/roberta-base-finetuned-ynat_bench        6/14  정확도 부족
#   마지막 두 줄 : 섹션이 애매한 제목은 확률이 낮게 나옴 -> 서비스의 '검토 필요' 기준(0.6)의 근거
# 섹션별 제목 2건씩, 직접 만든 14건으로 정확도와 속도를 확인합니다
tests = [
    ("IT과학", "삼성전자, 차세대 AI 반도체 HBM4 양산 돌입"), ("IT과학", "카카오, 생성형 AI 챗봇 서비스 정식 출시"),
    ("경제", "한국은행 기준금리 0.25%p 인하…가계대출 부담 줄어"), ("경제", "코스피 2,800선 회복, 외국인 순매수 이어져"),
    ("사회", "서울 지하철 노조 내일 총파업 예고…출근길 혼잡 우려"), ("사회", "경찰, 보이스피싱 조직 총책 검거"),
    ("생활문화", "올가을 단풍 절정 이번 주말…설악산 인파 몰려"), ("생활문화", "국립중앙박물관 특별전 관람객 10만 명 돌파"),
    ("세계", "미국 대선 앞두고 경합주 여론조사 접전"), ("세계", "일본 도쿄서 규모 5.8 지진…인명 피해 없어"),
    ("스포츠", "손흥민 시즌 10호 골…토트넘 3연승"), ("스포츠", "한국 야구대표팀, 프리미어12 결승 진출"),
    ("정치", "여야, 내년도 예산안 처리 두고 국회 대치"), ("정치", "대통령실, 개각 발표…국무총리 후보자 지명"),
]

ok, times = 0, []
for gold, title in tests:
    t0 = time.perf_counter()
    top = classifier(title)[0][0]
    times.append(time.perf_counter() - t0)
    ok += top["label"] == gold
    mark = "O" if top["label"] == gold else "X"
    print(f"  {mark} 정답 {gold:5} 예측 {top['label']:5} ({top['score']:.2f})  {title}")

print(f"\n정답 {ok}/{len(tests)}  평균 {sum(times) / len(times) * 1000:.0f}ms/건")

# 섹션이 애매한 제목은 확률이 낮게 나옵니다 -> 서비스에서 '편집자 검토 필요' 로 표시할 근거
for title in ["정부, AI 스타트업 육성에 1조 원 투입", "서울시장, 한강 리버버스 운항 개시"]:
    top2 = classifier(title)[0][:2]
    print(f"  애매 : {title} -> {[(s['label'], round(s['score'], 2)) for s in top2]}")

  O 정답 IT과학  예측 IT과학  (0.92)  삼성전자, 차세대 AI 반도체 HBM4 양산 돌입


  O 정답 IT과학  예측 IT과학  (0.98)  카카오, 생성형 AI 챗봇 서비스 정식 출시


  O 정답 경제    예측 경제    (0.99)  한국은행 기준금리 0.25%p 인하…가계대출 부담 줄어


  O 정답 경제    예측 경제    (0.99)  코스피 2,800선 회복, 외국인 순매수 이어져


  O 정답 사회    예측 사회    (0.98)  서울 지하철 노조 내일 총파업 예고…출근길 혼잡 우려


  O 정답 사회    예측 사회    (0.98)  경찰, 보이스피싱 조직 총책 검거


... (6 outputs left out)

  O 정답 정치    예측 정치    (0.99)  여야, 내년도 예산안 처리 두고 국회 대치


  O 정답 정치    예측 정치    (0.99)  대통령실, 개각 발표…국무총리 후보자 지명

정답 14/14  평균 1502ms/건


  애매 : 정부, AI 스타트업 육성에 1조 원 투입 -> [('IT과학', 0.55), ('정치', 0.31)]


  애매 : 서울시장, 한강 리버버스 운항 개시 -> [('사회', 0.58), ('생활문화', 0.4)]


In [4]:
# ==========================================================================
# [3. 프로젝트 뼈대] 폴더를 만들고 구조를 출력합니다
#   auth.py(재사용) -> schemas.py -> model_service.py -> main.py -> frontend/app.py 순서로 작성
# ==========================================================================
import os

dirs = ["app", "models", "frontend", "screenshots"]
for d in dirs:
    os.makedirs(d, exist_ok=True)
open("app/__init__.py", "a").close()   # app 을 패키지로 인식시킵니다 (uvicorn app.main:app)

print("프로젝트 구조:")
print("""
my-project/
├── 📁 app/
│   ├── __init__.py
│   ├── auth.py              ← Day 6 인증 모듈 재사용 (test-key-001, test-key-002)
│   ├── schemas.py           ← 제목 1건 / 일괄(최대 20건) 요청·응답 스키마
│   ├── model_service.py     ← 모델 로드 + 단건·일괄 추론 + 검토 필요 판정
│   └── main.py              ← FastAPI : /health, /labels, /predict, /predict/batch
│
├── 📁 frontend/
│   └── app.py               ← Streamlit : 한 건 분류 탭 + 일괄 분류 탭(CSV 업로드·내려받기)
│
├── 📁 screenshots/          ← 제출용 캡처
├── SUBMISSION.md            ← 제출 문서
└── requirements.txt
""")

프로젝트 구조:

my-project/
├── 📁 app/
│   ├── __init__.py
│   ├── auth.py              ← Day 6 인증 모듈 재사용 (test-key-001, test-key-002)
│   ├── schemas.py           ← 제목 1건 / 일괄(최대 20건) 요청·응답 스키마
│   ├── model_service.py     ← 모델 로드 + 단건·일괄 추론 + 검토 필요 판정
│   └── main.py              ← FastAPI : /health, /labels, /predict, /predict/batch
│
├── 📁 frontend/
│   └── app.py               ← Streamlit : 한 건 분류 탭 + 일괄 분류 탭(CSV 업로드·내려받기)
│
├── 📁 screenshots/          ← 제출용 캡처
├── SUBMISSION.md            ← 제출 문서
└── requirements.txt



In [5]:
%%writefile app/auth.py
# [3.2 app/auth.py] Day 6 인증 모듈 재사용 : X-API-Key 헤더가 없거나 틀리면 401
"""
Day 6에서 만든 인증 모듈을 그대로 재사용합니다.
"""
from fastapi import HTTPException, Header

VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    return VALID_API_KEYS[x_api_key]

Overwriting app/auth.py


In [6]:
%%writefile app/schemas.py
# [3.3 app/schemas.py] 입력/출력 규격 : 위반하면 추론 전에 FastAPI 가 422 로 거절
#   제목 2~200자(공백만 금지), top_k 1~7, 일괄 1~20건
"""
뉴스 섹션 분류 API 의 입력/출력 스키마.

모델(KLUE-YNAT)은 '뉴스 제목' 으로 학습되었습니다. 그래서 입력도 제목 길이에 맞춰
짧게 제한합니다. 본문 전체를 넣으면 길이만 늘고 정확도는 오르지 않습니다.
"""
from pydantic import BaseModel, Field, field_validator

# 학습 데이터(KLUE-YNAT)의 7개 섹션. /labels 와 화면에서 함께 씁니다
SECTIONS = ["IT과학", "경제", "사회", "생활문화", "세계", "스포츠", "정치"]

MAX_TITLE_LEN = 200      # 제목 하나의 최대 글자 수
MAX_BATCH = 20           # 한 번에 분류할 수 있는 제목 수


def _not_blank(value: str) -> str:
    """공백만 있는 문자열은 min_length 를 통과하므로 따로 막습니다."""
    value = value.strip()
    if not value:
        raise ValueError("공백만으로 된 제목은 분류할 수 없습니다.")
    return value


# ===== 요청 =====

class PredictRequest(BaseModel):
    """뉴스 제목 한 건 분류 요청"""
    title: str = Field(
        ..., min_length=2, max_length=MAX_TITLE_LEN,
        description="분류할 뉴스 제목 (2~200자)",
    )
    top_k: int = Field(
        default=3, ge=1, le=len(SECTIONS),
        description="함께 돌려받을 상위 후보 수 (1~7)",
    )

    @field_validator("title")
    @classmethod
    def strip_title(cls, value: str) -> str:
        return _not_blank(value)

    model_config = {
        "json_schema_extra": {
            "examples": [{"title": "삼성전자, 차세대 AI 반도체 양산 돌입", "top_k": 3}]
        }
    }


class BatchPredictRequest(BaseModel):
    """뉴스 제목 여러 건 일괄 분류 요청"""
    titles: list[str] = Field(
        ..., min_length=1, max_length=MAX_BATCH,
        description=f"분류할 뉴스 제목 목록 (1~{MAX_BATCH}건)",
    )

    @field_validator("titles")
    @classmethod
    def check_each_title(cls, titles: list[str]) -> list[str]:
        cleaned = []
        for i, t in enumerate(titles, start=1):
            t = t.strip()
            if len(t) < 2 or len(t) > MAX_TITLE_LEN:
                raise ValueError(f"{i}번째 제목의 길이가 2~{MAX_TITLE_LEN}자를 벗어났습니다.")
            cleaned.append(t)
        return cleaned

    model_config = {
        "json_schema_extra": {
            "examples": [{"titles": [
                "한국은행 기준금리 동결",
                "손흥민 시즌 10호 골",
                "여야, 예산안 두고 국회 대치",
            ]}]
        }
    }


# ===== 응답 =====

class SectionScore(BaseModel):
    """섹션 후보 하나와 그 확률"""
    section: str = Field(description="뉴스 섹션")
    score: float = Field(description="확률 (0~1)")


class PredictResponse(BaseModel):
    """한 건 분류 결과"""
    success: bool = True
    title: str = Field(description="분류한 제목 (앞뒤 공백 제거)")
    section: str = Field(description="가장 가능성 높은 섹션")
    confidence: float = Field(description="그 섹션의 확률 (0~1)")
    needs_review: bool = Field(description="확률이 기준보다 낮아 편집자 확인이 필요한지")
    candidates: list[SectionScore] = Field(description="상위 top_k 후보")
    user: str = Field(description="요청한 사용자 (API Key 소유자)")
    elapsed_ms: float = Field(description="추론에 걸린 시간 (밀리초)")


class BatchItem(BaseModel):
    title: str
    section: str
    confidence: float
    needs_review: bool


class BatchPredictResponse(BaseModel):
    """일괄 분류 결과"""
    success: bool = True
    count: int = Field(description="분류한 제목 수")
    results: list[BatchItem]
    section_counts: dict[str, int] = Field(description="섹션별 건수")
    review_count: int = Field(description="확인이 필요한 제목 수")
    user: str
    elapsed_ms: float

Overwriting app/schemas.py


In [7]:
%%writefile app/model_service.py
# [3.4 app/model_service.py] 모델 로드 + 단건/일괄 추론 + '검토 필요' 판정 (1위 확률 < 0.6)
"""
모델 로드와 추론 함수.

모델 : chunwoolee0/klue_ynat_roberta_base_model (KLUE-RoBERTa-base, KLUE-YNAT 7개 섹션)
- 원본 저장소는 pytorch_model.bin 만 있어 torch 2.6 미만에서는 읽히지 않습니다.
  (transformers 가 torch.load 취약점 CVE-2025-32434 때문에 막음)
- 허깅페이스 변환 봇이 올린 safetensors 판(refs/pr/2)을 revision 으로 지정해 해결합니다.

모델 이름, revision, 검토 기준값은 환경 변수로 바꿀 수 있습니다.
"""
import os
import time

from transformers import pipeline

MODEL_ID = os.environ.get("NEWS_MODEL_ID", "chunwoolee0/klue_ynat_roberta_base_model")
MODEL_REVISION = os.environ.get("NEWS_MODEL_REVISION", "refs/pr/2")

# 1위 확률이 이 값보다 낮으면 '편집자 검토 필요' 로 표시합니다
REVIEW_THRESHOLD = float(os.environ.get("NEWS_REVIEW_THRESHOLD", "0.6"))


def load_model():
    """분류 파이프라인을 만듭니다. 서버 시작 시 한 번만 부릅니다."""
    return pipeline(
        "text-classification",
        model=MODEL_ID,
        revision=MODEL_REVISION,
        top_k=None,          # 7개 섹션 확률을 모두 받습니다 (확률 내림차순)
        device=-1,           # CPU
    )


def _to_result(title: str, scores: list[dict], top_k: int) -> dict:
    """파이프라인 출력 한 건을 서비스 응답 형태로 바꿉니다."""
    best = scores[0]
    return {
        "title": title,
        "section": best["label"],
        "confidence": round(best["score"], 4),
        "needs_review": best["score"] < REVIEW_THRESHOLD,
        "candidates": [
            {"section": s["label"], "score": round(s["score"], 4)}
            for s in scores[:top_k]
        ],
    }


def predict(model, title: str, top_k: int = 3) -> dict:
    """제목 한 건을 분류합니다. (동기 함수 -> 서버에서 별도 스레드로 실행)"""
    start = time.perf_counter()
    scores = model(title, truncation=True)[0]
    result = _to_result(title, scores, top_k)
    result["elapsed_ms"] = round((time.perf_counter() - start) * 1000, 1)
    return result


def predict_batch(model, titles: list[str]) -> dict:
    """제목 여러 건을 한 번에 분류합니다. 파이프라인에 리스트로 넘겨 배치로 처리합니다."""
    start = time.perf_counter()
    outputs = model(titles, truncation=True, batch_size=8)
    results = []
    counts: dict[str, int] = {}
    for title, scores in zip(titles, outputs):
        r = _to_result(title, scores, top_k=1)
        results.append({k: r[k] for k in ("title", "section", "confidence", "needs_review")})
        counts[r["section"]] = counts.get(r["section"], 0) + 1
    return {
        "count": len(results),
        "results": results,
        "section_counts": dict(sorted(counts.items(), key=lambda kv: -kv[1])),
        "review_count": sum(r["needs_review"] for r in results),
        "elapsed_ms": round((time.perf_counter() - start) * 1000, 1),
    }

Overwriting app/model_service.py


In [8]:
%%writefile app/main.py
# [3.5 app/main.py] FastAPI 서버 조립
#   lifespan 에서 모델 1회 로드 / Depends(verify_api_key) 인증 / run_in_executor 비동기 추론
#   상태 코드 : 200 정상, 401 키 없음·틀림, 422 입력 위반, 503 모델 로드 전, 500 내부 오류(정보 숨김)
#   (startup 이벤트 대신 lifespan 을 썼습니다. 둘 다 가능)
"""
뉴스 섹션 자동 분류 FastAPI 서버.

엔드포인트
  GET  /health          서버/모델 상태 (인증 없음 : 게이트웨이·모니터링용)
  GET  /labels          분류 가능한 섹션 목록 (인증 없음)
  POST /predict         제목 한 건 분류 (API Key 필요)
  POST /predict/batch   제목 여러 건 일괄 분류 (API Key 필요)

상태 코드
  200 정상 / 401 API Key 없음·틀림 / 422 입력 규격 위반 / 503 모델 준비 전 / 500 추론 실패
"""
import asyncio
import logging
import time
from concurrent.futures import ThreadPoolExecutor
from contextlib import asynccontextmanager

from fastapi import Depends, FastAPI, HTTPException, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse

from app.auth import verify_api_key
from app.model_service import (MODEL_ID, MODEL_REVISION, REVIEW_THRESHOLD,
                               load_model, predict, predict_batch)
from app.schemas import (SECTIONS, BatchPredictRequest, BatchPredictResponse,
                         PredictRequest, PredictResponse)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s [news_api] %(message)s")
logger = logging.getLogger("news_api")

# 추론 전용 스레드풀 : 무거운 동기 추론이 이벤트 루프를 막지 않게 합니다
inference_executor = ThreadPoolExecutor(max_workers=2, thread_name_prefix="news-infer")

STATE = {"model": None, "loaded_at": None, "load_seconds": None}


@asynccontextmanager
async def lifespan(app: FastAPI):
    # 시작 : 모델을 한 번만 올립니다. 다운로드·로드도 무거워서 스레드에서 돌립니다
    logger.info(f"모델 로드 시작 : {MODEL_ID} ({MODEL_REVISION})")
    start = time.perf_counter()
    loop = asyncio.get_running_loop()
    try:
        STATE["model"] = await loop.run_in_executor(inference_executor, load_model)
        STATE["load_seconds"] = round(time.perf_counter() - start, 1)
        STATE["loaded_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
        logger.info(f"모델 로드 완료 ({STATE['load_seconds']}초)")
    except Exception:
        # 로드 실패로 서버가 죽지 않게 합니다. /health 가 loading 을, /predict 가 503 을 알립니다
        logger.exception("모델 로드 실패")
    yield
    # 종료 : 스레드풀 정리
    inference_executor.shutdown(wait=False, cancel_futures=True)
    logger.info("서버 종료")


app = FastAPI(
    title="뉴스 섹션 자동 분류 API",
    description=(
        "뉴스 제목을 받아 KLUE-YNAT 7개 섹션(IT과학·경제·사회·생활문화·세계·스포츠·정치) "
        "중 하나로 분류합니다. 확률이 낮은 제목은 `needs_review` 로 표시해 편집자 확인을 돕습니다."
    ),
    version="1.0.0",
    lifespan=lifespan,
)


# ===== 미들웨어 : 처리 시간 헤더 + 요청 로그 =====
@app.middleware("http")
async def add_process_time(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    elapsed = (time.perf_counter() - start) * 1000
    response.headers["X-Process-Time-ms"] = f"{elapsed:.1f}"
    logger.info(f"{request.method} {request.url.path} -> {response.status_code} ({elapsed:.0f}ms)")
    return response


# ===== 에러 핸들러 =====
@app.exception_handler(RequestValidationError)
async def validation_error_handler(request: Request, exc: RequestValidationError):
    """422 응답을 사람이 읽기 쉬운 형태로 바꿉니다. (어느 필드가 왜 틀렸는지)"""
    errors = [
        {
            # 본문이 JSON 이 아니면 loc 이 ("body", 위치) 라서 필드 이름 대신 body 로 표시합니다
            "field": "body" if e["type"] == "json_invalid"
            else ".".join(str(p) for p in e["loc"][1:]) or "body",
            "message": e["msg"],
        }
        for e in exc.errors()
    ]
    return JSONResponse(
        status_code=422,
        content={"success": False, "error": "입력값이 올바르지 않습니다.", "details": errors},
    )


@app.exception_handler(Exception)
async def unhandled_error_handler(request: Request, exc: Exception):
    """예상 못 한 에러 : 상세는 로그에만 남기고 요청자에게는 내부 정보를 숨깁니다."""
    logger.exception(f"처리되지 않은 에러 : {request.method} {request.url.path}")
    return JSONResponse(
        status_code=500,
        content={"success": False, "error": "서버 내부 오류가 발생했습니다."},
    )


def _require_model():
    if STATE["model"] is None:
        raise HTTPException(status_code=503, detail="모델을 불러오는 중입니다. 잠시 후 다시 시도해 주세요.")
    return STATE["model"]


# ===== 엔드포인트 =====
@app.get("/health", tags=["System"])
async def health():
    ready = STATE["model"] is not None
    return {
        "status": "healthy" if ready else "loading",
        "model": MODEL_ID,
        "revision": MODEL_REVISION,
        "loaded_at": STATE["loaded_at"],
        "load_seconds": STATE["load_seconds"],
    }


@app.get("/labels", tags=["System"])
async def labels():
    return {"sections": SECTIONS, "review_threshold": REVIEW_THRESHOLD}


@app.post("/predict", response_model=PredictResponse, tags=["Prediction"])
async def predict_one(req: PredictRequest, user: str = Depends(verify_api_key)):
    """뉴스 제목 한 건을 분류합니다."""
    model = _require_model()
    loop = asyncio.get_running_loop()
    try:
        result = await loop.run_in_executor(inference_executor, predict, model, req.title, req.top_k)
    except Exception as e:
        logger.exception("추론 실패")
        raise HTTPException(status_code=500, detail=f"추론에 실패했습니다 ({type(e).__name__}).")
    return PredictResponse(success=True, user=user, **result)


@app.post("/predict/batch", response_model=BatchPredictResponse, tags=["Prediction"])
async def predict_many(req: BatchPredictRequest, user: str = Depends(verify_api_key)):
    """뉴스 제목 여러 건(최대 20건)을 한 번에 분류합니다."""
    model = _require_model()
    loop = asyncio.get_running_loop()
    try:
        result = await loop.run_in_executor(inference_executor, predict_batch, model, req.titles)
    except Exception as e:
        logger.exception("일괄 추론 실패")
        raise HTTPException(status_code=500, detail=f"추론에 실패했습니다 ({type(e).__name__}).")
    return BatchPredictResponse(success=True, user=user, **result)

Overwriting app/main.py


In [9]:
%%writefile frontend/app.py
# [3.6 frontend/app.py] Streamlit 화면
#   사이드바 : API Key 입력, 서버 상태 / 탭 1 : 제목 한 건 분류 / 탭 2 : 일괄 분류(CSV 업로드·내려받기)
#   실행 : streamlit run frontend/app.py  -> http://localhost:8501
"""
뉴스 섹션 자동 분류 화면 (Streamlit).

실행 : streamlit run frontend/app.py   (FastAPI 서버가 먼저 떠 있어야 합니다)
"""
import os

import pandas as pd
import requests
import streamlit as st

API_BASE = os.environ.get("NEWS_API_BASE", "http://localhost:8000")

SAMPLE_TITLES = [
    "삼성전자, 차세대 AI 반도체 HBM4 양산 돌입",
    "한국은행 기준금리 0.25%p 인하…가계대출 부담 줄어",
    "서울 지하철 노조 내일 총파업 예고…출근길 혼잡 우려",
    "올가을 단풍 절정 이번 주말…설악산 인파 몰려",
    "일본 도쿄서 규모 5.8 지진…인명 피해 없어",
    "손흥민 시즌 10호 골…토트넘 3연승",
    "여야, 내년도 예산안 처리 두고 국회 대치",
    "정부, AI 스타트업 육성에 1조 원 투입",        # IT과학/정치 사이 -> 검토 필요 예시
    "서울시장, 한강 리버버스 운항 개시",            # 사회/생활문화 사이 -> 검토 필요 예시
]

SECTION_ICON = {
    "IT과학": "💻", "경제": "💰", "사회": "🏙️", "생활문화": "🎨",
    "세계": "🌏", "스포츠": "⚽", "정치": "🏛️",
}

st.set_page_config(page_title="뉴스 섹션 자동 분류", page_icon="📰", layout="wide")


# ===== API 호출 =====
def call_api(path, payload=None, api_key=None, method="post"):
    """API 를 호출하고 (데이터, 에러 메시지) 를 돌려줍니다. 상태 코드별로 안내 문구를 다르게 씁니다."""
    headers = {"X-API-Key": api_key} if api_key else {}
    try:
        if method == "get":
            resp = requests.get(f"{API_BASE}{path}", timeout=10)
        else:
            resp = requests.post(f"{API_BASE}{path}", json=payload, headers=headers, timeout=60)
    except requests.exceptions.ConnectionError:
        return None, "🔌 서버에 연결할 수 없습니다. FastAPI 서버가 실행 중인지 확인해 주세요."
    except requests.exceptions.Timeout:
        return None, "⏱️ 응답 시간이 초과되었습니다. 잠시 후 다시 시도해 주세요."

    if resp.status_code == 200:
        return resp.json(), None
    body = resp.json() if resp.headers.get("content-type", "").startswith("application/json") else {}
    if resp.status_code == 401:
        return None, f"🔑 인증 실패 (401) : {body.get('detail', '')} 사이드바의 API Key 를 확인해 주세요."
    if resp.status_code == 422:
        lines = [f"- `{d['field']}` : {d['message']}" for d in body.get("details", [])]
        return None, "✏️ 입력값 오류 (422)\n" + "\n".join(lines)
    if resp.status_code == 503:
        return None, "⏳ 모델을 불러오는 중입니다 (503). 잠시 후 다시 시도해 주세요."
    return None, f"❌ 서버 오류 (HTTP {resp.status_code}) : {body.get('error') or body.get('detail', '')}"


# ===== 사이드바 =====
with st.sidebar:
    st.header("⚙️ 설정")
    api_key = st.text_input("API Key", value="test-key-001", type="password",
                            help="실습용 키 : test-key-001, test-key-002")

    health, err = call_api("/health", method="get")
    if health and health["status"] == "healthy":
        st.success("🟢 서버 연결됨 · 모델 준비 완료")
        st.caption(f"모델 : `{health['model']}`")
        st.caption(f"로드 시각 : {health['loaded_at']} ({health['load_seconds']}초)")
    elif health:
        st.warning("🟡 서버는 떠 있지만 모델을 불러오는 중입니다.")
    else:
        st.error("🔴 서버 연결 실패")

    labels, _ = call_api("/labels", method="get")
    if labels:
        st.divider()
        st.caption("분류 가능한 섹션")
        st.write(" ".join(f"{SECTION_ICON.get(s, '')}{s}" for s in labels["sections"]))
        st.caption(f"확률 {labels['review_threshold']:.0%} 미만은 '검토 필요' 로 표시합니다.")


# ===== 메인 =====
st.title("📰 뉴스 섹션 자동 분류")
st.write("기사 제목을 넣으면 KLUE-YNAT 7개 섹션 중 어디에 실을지 추천합니다. "
         "확신이 낮은 제목은 편집자가 확인하도록 표시합니다.")

tab_one, tab_batch = st.tabs(["📝 제목 한 건 분류", "📋 여러 건 일괄 분류"])

# ----- 탭 1 : 한 건 -----
with tab_one:
    col_in, col_out = st.columns([1, 1])
    with col_in:
        sample = st.selectbox("예시 제목 (선택하면 아래 입력란에 채워집니다)", ["직접 입력"] + SAMPLE_TITLES)
        title = st.text_input("기사 제목", value="" if sample == "직접 입력" else sample,
                              max_chars=200, placeholder="예) 코스피 2,800선 회복, 외국인 순매수 이어져")
        top_k = st.slider("후보 섹션 수 (top_k)", 1, 7, 3)
        run_one = st.button("🔍 분류하기", type="primary", width="stretch")

    with col_out:
        if run_one:
            with st.spinner("분류 중..."):
                data, err = call_api("/predict", {"title": title, "top_k": top_k}, api_key)
            st.session_state["one_result"] = (data, err)

        if "one_result" in st.session_state:
            data, err = st.session_state["one_result"]
            if err:
                st.error(err)
            else:
                icon = SECTION_ICON.get(data["section"], "")
                c1, c2 = st.columns(2)
                c1.metric("추천 섹션", f"{icon} {data['section']}")
                c2.metric("확률", f"{data['confidence']:.1%}")
                if data["needs_review"]:
                    st.warning("⚠️ 확신이 낮습니다. 편집자 확인이 필요합니다.")
                else:
                    st.success("✅ 자동 배정 가능")
                chart = pd.DataFrame(data["candidates"]).set_index("section")
                st.bar_chart(chart, horizontal=True)
                st.caption(f"요청자 : {data['user']} · 추론 {data['elapsed_ms']}ms")

# ----- 탭 2 : 여러 건 -----
with tab_batch:
    st.write("한 줄에 제목 하나씩 입력하거나, `title` 열이 있는 CSV 를 올립니다. (최대 20건)")
    text = st.text_area("기사 제목 목록", value="\n".join(SAMPLE_TITLES), height=200)
    uploaded = st.file_uploader("또는 CSV 업로드", type=["csv"])

    if st.button("📋 일괄 분류", type="primary"):
        if uploaded is not None:
            try:
                df_in = pd.read_csv(uploaded)
                titles = df_in["title"].dropna().astype(str).tolist()
            except Exception:
                titles = None
                st.error("CSV 를 읽지 못했습니다. `title` 열이 있는지 확인해 주세요.")
        else:
            titles = [line for line in text.splitlines() if line.strip()]

        if titles is not None:
            with st.spinner(f"{len(titles)}건 분류 중..."):
                data, err = call_api("/predict/batch", {"titles": titles}, api_key)
            st.session_state["batch_result"] = (data, err)

    if "batch_result" in st.session_state:
        data, err = st.session_state["batch_result"]
        if err:
            st.error(err)
        else:
            c1, c2, c3 = st.columns(3)
            c1.metric("분류한 제목", f"{data['count']}건")
            c2.metric("검토 필요", f"{data['review_count']}건")
            c3.metric("처리 시간", f"{data['elapsed_ms'] / 1000:.1f}초")

            df = pd.DataFrame(data["results"])
            df["confidence"] = (df["confidence"] * 100).round(1)
            df["needs_review"] = df["needs_review"].map({True: "⚠️ 검토", False: "✅"})
            df.columns = ["제목", "섹션", "확률(%)", "상태"]

            left, right = st.columns([2, 1])
            left.dataframe(df, width="stretch", hide_index=True)
            right.write("**섹션별 건수**")
            right.bar_chart(pd.Series(data["section_counts"], name="건수"))
            st.download_button("⬇️ 결과 CSV 내려받기", df.to_csv(index=False).encode("utf-8-sig"),
                               file_name="news_sections.csv", mime="text/csv")

Overwriting frontend/app.py


In [10]:
# ==========================================================================
# [4.2 서버 실행] 노트북 안에서 uvicorn 을 백그라운드 스레드로 띄웁니다
#   권장 순서 : 모델 확인 -> 스키마 -> 추론 모듈 -> 서버 -> Swagger 테스트 -> Streamlit
#   모델 로드에 몇 초 걸리므로 /health 가 healthy 가 될 때까지 기다립니다
# ==========================================================================
# ⚠️ 커널 재시작 후 실행하세요 (같은 커널에서 두 번 띄우면 8000번이 겹칩니다)

import nest_asyncio, uvicorn, threading, time, requests
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# sleep(3) 대신 모델 로드가 끝날 때까지 /health 를 기다립니다
for _ in range(60):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).json()["status"] == "healthy":
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)

print("✅ 서버 시작됨 — http://localhost:8000/docs")

INFO:     Started server process [325120]


INFO:     Waiting for application startup.


2026-11-04 16:54:30,976 INFO [news_api] 모델 로드 시작 : chunwoolee0/klue_ynat_roberta_base_model (refs/pr/2)


Device set to use cpu


2026-11-04 16:54:31,806 INFO [news_api] 모델 로드 완료 (0.8초)


INFO:     Application startup complete.


INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


2026-11-04 16:54:32,944 INFO [news_api] GET /health -> 200 (6ms)


INFO:     127.0.0.1:43938 - "GET /health HTTP/1.1" 200 OK


✅ 서버 시작됨 — http://localhost:8000/docs


In [11]:
# ==========================================================================
# [4.3 API 테스트 1] 헬스체크 + 정상 추론 (200)
# ==========================================================================
import json
import requests

API_URL = "http://localhost:8000"
HEADERS = {"X-API-Key": "test-key-001"}

def show(resp):
    print(f"상태: {resp.status_code}")
    print(json.dumps(resp.json(), ensure_ascii=False, indent=2))

# health check
print(requests.get(f"{API_URL}/health").json())

# 추론 테스트 — 정상 (200)
response = requests.post(
    f"{API_URL}/predict",
    json={"title": "코스피 2,800선 회복, 외국인 순매수 이어져", "top_k": 3},
    headers=HEADERS,
)
show(response)

2026-11-04 16:54:32,953 INFO [news_api] GET /health -> 200 (0ms)


INFO:     127.0.0.1:43952 - "GET /health HTTP/1.1" 200 OK


2026-11-04 16:54:33,040 INFO [news_api] POST /predict -> 200 (84ms)


{'status': 'healthy', 'model': 'chunwoolee0/klue_ynat_roberta_base_model', 'revision': 'refs/pr/2', 'loaded_at': '2026-11-04 16:54:31', 'load_seconds': 0.8}
INFO:     127.0.0.1:43968 - "POST /predict HTTP/1.1" 200 OK


상태: 200
{
  "success": true,
  "title": "코스피 2,800선 회복, 외국인 순매수 이어져",
  "section": "경제",
  "confidence": 0.9907,
  "needs_review": false,
  "candidates": [
    {
      "section": "경제",
      "score": 0.9907
    },
    {
      "section": "사회",
      "score": 0.0021
    },
    {
      "section": "세계",
      "score": 0.0021
    }
  ],
  "user": "사용자A",
  "elapsed_ms": 81.8
}


In [12]:
# [4.3 API 테스트 2] 검토 필요 사례 : 1위 확률이 기준(60%)보다 낮으면 needs_review: true
show(requests.post(f"{API_URL}/predict",
                   json={"title": "정부, AI 스타트업 육성에 1조 원 투입", "top_k": 2},
                   headers=HEADERS))

2026-11-04 16:54:33,104 INFO [news_api] POST /predict -> 200 (56ms)


INFO:     127.0.0.1:43982 - "POST /predict HTTP/1.1" 200 OK


상태: 200
{
  "success": true,
  "title": "정부, AI 스타트업 육성에 1조 원 투입",
  "section": "IT과학",
  "confidence": 0.5506,
  "needs_review": true,
  "candidates": [
    {
      "section": "IT과학",
      "score": 0.5506
    },
    {
      "section": "정치",
      "score": 0.3121
    }
  ],
  "user": "사용자A",
  "elapsed_ms": 54.5
}


In [13]:
# [4.3 API 테스트 3] 인증 실패 (401) : 키가 없을 때와 틀렸을 때
body = {"title": "테스트용 뉴스 제목"}
r1 = requests.post(f"{API_URL}/predict", json=body)
r2 = requests.post(f"{API_URL}/predict", json=body, headers={"X-API-Key": "wrong-key"})
print(f"키 없음   -> {r1.status_code} {r1.json()}")
print(f"틀린 키   -> {r2.status_code} {r2.json()}")

2026-11-04 16:54:33,112 INFO [news_api] POST /predict -> 401 (1ms)


INFO:     127.0.0.1:43996 - "POST /predict HTTP/1.1" 401 Unauthorized


2026-11-04 16:54:33,116 INFO [news_api] POST /predict -> 401 (1ms)


INFO:     127.0.0.1:44010 - "POST /predict HTTP/1.1" 401 Unauthorized


키 없음   -> 401 {'detail': 'API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.'}
틀린 키   -> 401 {'detail': '유효하지 않은 API Key입니다.'}


In [14]:
# [4.3 API 테스트 4] 잘못된 입력 (422) : 어느 필드가 왜 틀렸는지 한국어 메시지로 돌려줌
cases = [
    ("필드 누락",        {}),
    ("한 글자 제목",     {"title": "가"}),
    ("공백만 있는 제목", {"title": "     "}),
    ("너무 긴 제목",     {"title": "가" * 201}),
    ("top_k 범위 초과",  {"title": "테스트용 뉴스 제목", "top_k": 9}),
]
for name, body in cases:
    r = requests.post(f"{API_URL}/predict", json=body, headers=HEADERS)
    print(f"{name:12} -> {r.status_code}  {r.json()['details']}")

r = requests.post(f"{API_URL}/predict", data="not json", headers=HEADERS)
print(f"{'JSON 아님':12} -> {r.status_code}  {r.json()['details']}")
r = requests.post(f"{API_URL}/predict/batch", json={"titles": ["뉴스 제목"] * 21}, headers=HEADERS)
print(f"{'일괄 21건':12} -> {r.status_code}  {r.json()['details']}")

2026-11-04 16:54:33,124 INFO [news_api] POST /predict -> 422 (1ms)


INFO:     127.0.0.1:44018 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


2026-11-04 16:54:33,127 INFO [news_api] POST /predict -> 422 (0ms)


필드 누락        -> 422  [{'field': 'title', 'message': 'Field required'}]
INFO:     127.0.0.1:44020 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


2026-11-04 16:54:33,131 INFO [news_api] POST /predict -> 422 (1ms)


한 글자 제목      -> 422  [{'field': 'title', 'message': 'String should have at least 2 characters'}]
INFO:     127.0.0.1:44030 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


... (5 outputs left out)

top_k 범위 초과  -> 422  [{'field': 'top_k', 'message': 'Input should be less than or equal to 7'}]
INFO:     127.0.0.1:44066 - "POST /predict HTTP/1.1" 422 Unprocessable Entity


2026-11-04 16:54:33,144 INFO [news_api] POST /predict/batch -> 422 (2ms)


JSON 아님      -> 422  [{'field': 'body', 'message': 'Input should be a valid dictionary or object to extract fields from'}]
INFO:     127.0.0.1:44068 - "POST /predict/batch HTTP/1.1" 422 Unprocessable Entity


일괄 21건       -> 422  [{'field': 'titles', 'message': 'List should have at most 20 items after validation, not 21'}]


In [15]:
# [4.3 API 테스트 5] 일괄 분류 POST /predict/batch : 섹션별 건수와 검토 필요 건수를 함께 반환
titles = [t for _, t in tests[::2]] + ["정부, AI 스타트업 육성에 1조 원 투입", "서울시장, 한강 리버버스 운항 개시"]
r = requests.post(f"{API_URL}/predict/batch", json={"titles": titles}, headers=HEADERS)
data = r.json()
print(f"상태 {r.status_code} · {data['count']}건 · 검토 필요 {data['review_count']}건 · {data['elapsed_ms']}ms")
for item in data["results"]:
    flag = "⚠️ 검토" if item["needs_review"] else "✅"
    print(f"  {flag} {item['section']:5} {item['confidence']:.2f}  {item['title']}")
print("섹션별 :", data["section_counts"])

2026-11-04 16:54:33,313 INFO [news_api] POST /predict/batch -> 200 (161ms)


INFO:     127.0.0.1:44080 - "POST /predict/batch HTTP/1.1" 200 OK


상태 200 · 9건 · 검토 필요 2건 · 159.1ms
  ✅ IT과학  0.92  삼성전자, 차세대 AI 반도체 HBM4 양산 돌입
  ✅ 경제    0.99  한국은행 기준금리 0.25%p 인하…가계대출 부담 줄어
  ✅ 사회    0.98  서울 지하철 노조 내일 총파업 예고…출근길 혼잡 우려
  ✅ 생활문화  0.99  올가을 단풍 절정 이번 주말…설악산 인파 몰려
  ✅ 세계    0.99  미국 대선 앞두고 경합주 여론조사 접전
  ✅ 스포츠   0.99  손흥민 시즌 10호 골…토트넘 3연승
  ✅ 정치    0.99  여야, 내년도 예산안 처리 두고 국회 대치
  ⚠️ 검토 IT과학  0.55  정부, AI 스타트업 육성에 1조 원 투입
  ⚠️ 검토 사회    0.58  서울시장, 한강 리버버스 운항 개시
섹션별 : {'IT과학': 2, '사회': 2, '경제': 1, '생활문화': 1, '세계': 1, '스포츠': 1, '정치': 1}


In [16]:
# [4.3 API 테스트 6] 동시 요청 : 추론은 run_in_executor 로 별도 스레드에서 돌아서
#   일괄 요청이 몰려도 이벤트 루프가 막히지 않고 /health 가 바로 응답합니다
from concurrent.futures import ThreadPoolExecutor

titles20 = [t for _, t in tests] + [t for _, t in tests[:6]]     # 20건짜리 일괄 요청

def one_batch(_):
    t0 = time.perf_counter()
    r = requests.post(f"{API_URL}/predict/batch", json={"titles": titles20}, headers=HEADERS, timeout=120)
    return r.status_code, time.perf_counter() - t0

t_start = time.perf_counter()
with ThreadPoolExecutor(max_workers=6) as ex:
    futures = [ex.submit(one_batch, i) for i in range(6)]     # 20건 x 6 = 120건을 동시에
    time.sleep(0.2)                                           # 추론이 한창 도는 중에 헬스체크
    t0 = time.perf_counter()
    h = requests.get(f"{API_URL}/health", timeout=10)
    health_at = t0 - t_start
    health_ms = (time.perf_counter() - t0) * 1000
    results = [f.result() for f in futures]

total = time.perf_counter() - t_start
print(f"일괄 6건(120제목) 상태 : {[s for s, _ in results]}  전체 {total:.1f}초")
print(f"추론 중({health_at:.1f}초 시점) /health : {h.status_code}, {health_ms:.0f}ms 만에 응답")

2026-11-04 16:54:33,533 INFO [news_api] GET /health -> 200 (1ms)


INFO:     127.0.0.1:44132 - "GET /health HTTP/1.1" 200 OK


2026-11-04 16:54:33,745 INFO [news_api] POST /predict/batch -> 200 (417ms)


INFO:     127.0.0.1:44092 - "POST /predict/batch HTTP/1.1" 200 OK


2026-11-04 16:54:33,752 INFO [news_api] POST /predict/batch -> 200 (425ms)


INFO:     127.0.0.1:44084 - "POST /predict/batch HTTP/1.1" 200 OK


... (5 outputs left out)

INFO:     127.0.0.1:44112 - "POST /predict/batch HTTP/1.1" 200 OK


2026-11-04 16:54:34,597 INFO [news_api] POST /predict/batch -> 200 (1266ms)


INFO:     127.0.0.1:44118 - "POST /predict/batch HTTP/1.1" 200 OK


일괄 6건(120제목) 상태 : [200, 200, 200, 200, 200, 200]  전체 1.3초
추론 중(0.2초 시점) /health : 200, 7ms 만에 응답


In [17]:
# [4.3 API 테스트 7] Streamlit 화면 확인 : 브라우저 없이 streamlit.testing 으로 버튼을 눌러 봅니다
#   실제 화면은 터미널에서 streamlit run frontend/app.py 로 띄워 확인 (캡처는 my-project/screenshots/)
from streamlit.testing.v1 import AppTest

import os
# 노트북에서는 상대 경로가 임시 폴더 기준으로 풀리므로 절대 경로로 넘깁니다
at = AppTest.from_file(os.path.abspath("frontend/app.py"), default_timeout=60).run()
print("사이드바 :", [s.value for s in at.sidebar.success])

at.selectbox[0].select("정부, AI 스타트업 육성에 1조 원 투입").run()
at.button[0].click().run()
print("한 건    :", [(m.label, m.value) for m in at.metric], [w.value for w in at.warning])

at.sidebar.text_input[0].set_value("wrong-key").run()
at.button[0].click().run()
print("401      :", [e.value for e in at.error])

at.sidebar.text_input[0].set_value("test-key-001").run()
at.button[1].click().run()
print("일괄     :", [(m.label, m.value) for m in at.metric][-3:])
print("화면 예외 :", list(at.exception))

2026-11-04 16:54:38.431 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-11-04 16:54:59,488 INFO [news_api] GET /health -> 200 (0ms)


INFO:     127.0.0.1:33056 - "GET /health HTTP/1.1" 200 OK


2026-11-04 16:54:59.489 127.0.0.1:33056 - "GET /health HTTP/1.1" 200


2026-11-04 16:54:59,495 INFO [news_api] GET /labels -> 200 (2ms)


INFO:     127.0.0.1:33066 - "GET /labels HTTP/1.1" 200 OK


... (43 outputs left out)

2026-11-04 16:55:01,311 INFO [news_api] POST /predict/batch -> 200 (161ms)


INFO:     127.0.0.1:47624 - "POST /predict/batch HTTP/1.1" 200 OK


2026-11-04 16:55:01.312 127.0.0.1:47624 - "POST /predict/batch HTTP/1.1" 200


일괄     : [('분류한 제목', '9건'), ('검토 필요', '2건'), ('처리 시간', '0.2초')]
화면 예외 : []


In [18]:
# ==========================================================================
# [5. 정리] 실습이 끝나면 서버와 Streamlit 을 종료합니다
#
# [발표 요약]
#   도메인·태스크 : 언론사 편집국, 기사 제목 -> 7개 섹션 자동 배정 (text-classification)
#   차별점       : ① 확률 60% 미만 편집자 검토 표시 ② 최대 20건 일괄 분류와 섹션별 집계
#                  ③ CSV 업로드·내려받기 ④ 422 오류를 필드별 한국어 메시지로 반환
#   어려웠던 점  : torch 2.4.1 에서 .bin 모델 로드 차단 -> safetensors PR 브랜치로 해결
#
# [체크포인트 답변]
#   1) 입력 검증 : 제목 2~200자, 공백만 금지(strip 후 재검사), top_k 1~7, 일괄 1~20건 -> 추론 전에 422
#   2) 인증 누락 : 주소만 알면 누구나 일괄 API 를 반복 호출해 CPU 를 점유할 수 있고 호출자도 남지 않음
#   3) 스레드 분리 : 동기 추론이 이벤트 루프를 막지 않게 함 (일괄 120제목 처리 중에도 /health 즉시 응답)
#   4) 참고 노트 : 정형 데이터 부록(서버 뼈대), 비동기 처리 노트(에러 핸들러), 인증과 화면 노트(Streamlit)
#   5) 운영 보완 : API 키 시크릿 분리·해시, 요청 제한, 모델 포함 Docker 이미지, 게이트웨이 헬스체크,
#                  검토 기준값 재조정, 모델 라이선스 확인(모델 카드에 표기 없음)
#
# [회고] 다음에는 모델을 고를 때 저장 형식(safetensors)과 torch 버전부터 확인하고,
#        검토 기준값은 검증 데이터를 더 모아 정밀도·재현율을 보고 정하겠습니다
# ==========================================================================
# 노트북 안에서 띄운 서버는 데몬 스레드라 커널을 종료하면 함께 내려갑니다
# 터미널에서 띄운 Streamlit 은 Ctrl+C 로 종료합니다
print("서버 스레드 동작 중 :", server_thread.is_alive(), "-> 커널 종료 시 함께 종료")

서버 스레드 동작 중 : True -> 커널 종료 시 함께 종료
